<a href="https://colab.research.google.com/github/Meenakshimadhu192001/TripLens/blob/main/TripLens_Full_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TripLens — Full Travel Package Preprocessing Pipeline

This notebook preprocesses the current TripLens workbook in the correct order:

1. Load raw workbook
2. Clean rows and remove schema/instruction rows
3. Standardize column names
4. Normalize missing values
5. Convert data types
6. Normalize booleans
7. Normalize numeric/range fields
8. Normalize categorical/list text
9. Normalize destinations
10. Clean itinerary data
11. Parse itinerary transit-hour ranges
12. Build itinerary-level features
13. Clean accommodation, inclusion and exclusion tables
14. Aggregate child tables to package level
15. Build cost and package-level features
16. Build canonical package text for NLP/embeddings
17. Validate the processed dataset
18. Save clean outputs

Important: this notebook does NOT generate embeddings or train the recommender yet.
The output `packages_for_embedding.csv` is the clean input for the next embedding/vector-search stage.

In [ ]:
# ============================================================
# 0. INSTALL / IMPORT LIBRARIES
# ============================================================

# If running in a fresh Colab environment, uncomment:
# !pip install -q pandas openpyxl numpy

import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

print("pandas:", pd.__version__)

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

# For the current uploaded workbook:
INPUT_FILE = "Package Details (1).xlsx"

# If your file is stored elsewhere in Colab, change INPUT_FILE.
# Example:
# INPUT_FILE = "/content/Package Details.xlsx"

OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_FILE)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. LOAD THE RAW WORKBOOK
# ============================================================

xls = pd.ExcelFile(INPUT_FILE)

print("Sheets found:")
for sheet in xls.sheet_names:
    print(" -", sheet)

# Load all relevant sheets.
raw_packages = pd.read_excel(INPUT_FILE, sheet_name="Packages")
raw_itinerary = pd.read_excel(INPUT_FILE, sheet_name="Itinerary_Days")
raw_accommodation = pd.read_excel(INPUT_FILE, sheet_name="Accommodation")
raw_inclusions = pd.read_excel(INPUT_FILE, sheet_name="Inclusions")
raw_exclusions = pd.read_excel(INPUT_FILE, sheet_name="Exclusions")

print("\nRaw shapes:")
print("Packages:", raw_packages.shape)
print("Itinerary:", raw_itinerary.shape)
print("Accommodation:", raw_accommodation.shape)
print("Inclusions:", raw_inclusions.shape)
print("Exclusions:", raw_exclusions.shape)

In [ ]:
# ============================================================
# 3. STANDARDIZE COLUMN NAMES
# ============================================================

def standardize_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

def standardize_columns(df):
    df = df.copy()
    df.columns = [standardize_column_name(c) for c in df.columns]
    return df

packages = standardize_columns(raw_packages)
itinerary = standardize_columns(raw_itinerary)
accommodation = standardize_columns(raw_accommodation)
inclusions = standardize_columns(raw_inclusions)
exclusions = standardize_columns(raw_exclusions)

# Fix the current workbook's typo:
if "cutomizable" in packages.columns:
    packages = packages.rename(columns={"cutomizable": "customizable"})

print("Packages columns:")
print(packages.columns.tolist())

In [ ]:
# ============================================================
# 4. REMOVE EMPTY ROWS AND TEMPLATE/INSTRUCTION ROWS
# ============================================================

def remove_empty_rows(df):
    df = df.copy()
    return df.dropna(how="all").reset_index(drop=True)

packages = remove_empty_rows(packages)
itinerary = remove_empty_rows(itinerary)
accommodation = remove_empty_rows(accommodation)
inclusions = remove_empty_rows(inclusions)
exclusions = remove_empty_rows(exclusions)

# The workbook contains a schema/instruction row directly below
# the real column headers. Remove rows whose package_id is not
# an actual package identifier.

PACKAGE_ID_PATTERN = r"^[A-Za-z]{2,5}\d+$"

def remove_instruction_rows(df):
    df = df.copy()
    if "package_id" in df.columns:
        valid = df["package_id"].astype("string").str.strip().str.match(
            PACKAGE_ID_PATTERN, na=False
        )
        df = df[valid].copy()
    return df.reset_index(drop=True)

packages = remove_instruction_rows(packages)
itinerary = remove_instruction_rows(itinerary)
accommodation = remove_instruction_rows(accommodation)
inclusions = remove_instruction_rows(inclusions)
exclusions = remove_instruction_rows(exclusions)

print("After removing empty/template rows:")
print("Packages:", packages.shape)
print("Itinerary:", itinerary.shape)
print("Accommodation:", accommodation.shape)
print("Inclusions:", inclusions.shape)
print("Exclusions:", exclusions.shape)

In [ ]:
# ============================================================
# 5. GENERAL TEXT CLEANING
# ============================================================

def clean_text(value):
    if pd.isna(value):
        return pd.NA

    text = str(value)

    # Normalize common whitespace characters.
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # Collapse repeated whitespace.
    text = re.sub(r"\s+", " ", text)

    # Remove spaces around separators without destroying meaning.
    text = re.sub(r"\s*;\s*", "; ", text)
    text = re.sub(r"\s*,\s*", ", ", text)

    text = text.strip()

    return text if text else pd.NA

def clean_text_columns(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].map(clean_text)
    return df

packages = clean_text_columns(packages)
itinerary = clean_text_columns(itinerary)
accommodation = clean_text_columns(accommodation)
inclusions = clean_text_columns(inclusions)
exclusions = clean_text_columns(exclusions)

print(packages.head())

In [ ]:
# ============================================================
# 6. STANDARDIZE MISSING VALUES
# ============================================================

MISSING_VALUES = {
    "", "na", "n/a", "nan", "none", "null",
    "not available", "not specified", "unknown"
}

def normalize_missing(df):
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == "object" or str(df[col].dtype).startswith("string"):
            df[col] = df[col].map(
                lambda x: pd.NA
                if pd.notna(x) and str(x).strip().lower() in MISSING_VALUES
                else x
            )

    return df

packages = normalize_missing(packages)
itinerary = normalize_missing(itinerary)
accommodation = normalize_missing(accommodation)
inclusions = normalize_missing(inclusions)
exclusions = normalize_missing(exclusions)

In [ ]:
# ============================================================
# 7. NORMALIZE BOOLEAN VALUES
# ============================================================

TRUE_VALUES = {"true", "yes", "y", "1", "t"}
FALSE_VALUES = {"false", "no", "n", "0", "f"}

def parse_boolean(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower()

    if text in TRUE_VALUES:
        return True

    if text in FALSE_VALUES:
        return False

    return pd.NA

if "tier_range_exists" in packages.columns:
    packages["tier_range_exists"] = packages["tier_range_exists"].map(parse_boolean)

if "customizable" in packages.columns:
    packages["customizable"] = packages["customizable"].map(parse_boolean)

def normalize_hotel_guaranteed(value):
    if pd.isna(value):
        return "unspecified"

    text = str(value).strip().lower()

    if text.startswith("true"):
        return "true"

    if text.startswith("false"):
        return "false"

    return "unspecified"

if "hotel_guaranteed" in accommodation.columns:
    accommodation["hotel_guaranteed"] = accommodation["hotel_guaranteed"].map(
        normalize_hotel_guaranteed
    )

In [ ]:
# ============================================================
# 8. NUMERIC CONVERSION
# ============================================================

def to_numeric(series):
    # Extract the first valid numeric value.
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False)
              .str.extract(r"(-?\d+(?:\.\d+)?)", expand=False),
        errors="coerce"
    )

for col in ["duration_days", "duration_nights", "price"]:
    if col in packages.columns:
        packages[col] = to_numeric(packages[col])

for col in ["day_number", "distance_km", "stops_requiring_separate_drives"]:
    if col in itinerary.columns:
        itinerary[col] = to_numeric(itinerary[col])

# Convert integer-like fields to nullable integer.
for col in ["duration_days", "duration_nights"]:
    if col in packages.columns:
        packages[col] = packages[col].round().astype("Int64")

if "day_number" in itinerary.columns:
    itinerary["day_number"] = itinerary["day_number"].round().astype("Int64")

if "stops_requiring_separate_drives" in itinerary.columns:
    itinerary["stops_requiring_separate_drives"] = (
        itinerary["stops_requiring_separate_drives"]
        .round()
        .astype("Int64")
    )

print(packages[["package_id", "duration_days", "duration_nights", "price"]])

In [ ]:
# ============================================================
# 9. PARSE TRANSIT-HOUR RANGES
# ============================================================

def parse_range(value):
    if pd.isna(value):
        return (np.nan, np.nan)

    text = str(value).lower().replace("hours", "").replace("hour", "")
    numbers = re.findall(r"\d+(?:\.\d+)?", text)

    if not numbers:
        return (np.nan, np.nan)

    nums = [float(x) for x in numbers]

    if len(nums) == 1:
        return (nums[0], nums[0])

    return (min(nums[0], nums[1]), max(nums[0], nums[1]))

if "transit_hours" in itinerary.columns:
    parsed_hours = itinerary["transit_hours"].map(parse_range)

    itinerary["transit_hours_min"] = parsed_hours.map(lambda x: x[0])
    itinerary["transit_hours_max"] = parsed_hours.map(lambda x: x[1])
    itinerary["transit_hours_avg"] = (
        itinerary["transit_hours_min"] +
        itinerary["transit_hours_max"]
    ) / 2

print(
    itinerary[
        ["package_id", "day_number", "transit_hours",
         "transit_hours_min", "transit_hours_max", "transit_hours_avg"]
    ]
)

In [ ]:
# ============================================================
# 10. NORMALIZE LIST/CATEGORY FIELDS
# ============================================================

def split_semicolon(value):
    if pd.isna(value):
        return []

    parts = [x.strip() for x in str(value).split(";")]
    return [x for x in parts if x]

def split_comma(value):
    if pd.isna(value):
        return []

    parts = [x.strip() for x in str(value).split(",")]
    return [x for x in parts if x]

# Destination list.
packages["destination_list"] = packages["destinations"].map(split_comma)

# Theme list.
packages["theme_list"] = packages["theme"].map(split_comma)

# Transport modes can be multiple.
packages["transport_list"] = packages["transport_type"].map(
    lambda x: split_semicolon(x)
)

# Clean standard spelling for known categories.
CATEGORY_MAP = {
    "hill-station": "hill_station",
    "hill station": "hill_station",
    "back water": "backwaters",
    "back-water": "backwaters",
    "eco tourism": "eco_tourism",
    "eco-tourism": "eco_tourism",
    "family trip": "family",
    "adventure tourism": "adventure"
}

def normalize_category(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)

    return CATEGORY_MAP.get(text, text)

packages["theme_list"] = packages["theme_list"].map(
    lambda items: [normalize_category(x) for x in items]
)

In [ ]:
# ============================================================
# 11. DESTINATION NORMALIZATION
# ============================================================

# Keep both the original destination string and a normalized list.
# We do NOT delete the original value.

def normalize_place(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)

    place_map = {
        "alleppey": "alappuzha",
        "trivandrum": "thiruvananthapuram",
        "new delhi": "delhi",
    }

    return place_map.get(text, text)

packages["destination_list_normalized"] = packages["destination_list"].map(
    lambda items: [normalize_place(x) for x in items]
)

packages["start_location_normalized"] = packages["start_location"].map(
    normalize_place
)

itinerary["stops_clean"] = itinerary["stops"].map(
    lambda x: [normalize_place(p) for p in re.split(r"\s*->\s*|\s*;\s*|\s*,\s*", str(x))]
    if pd.notna(x) else []
)

print(
    packages[
        ["package_id", "destinations", "destination_list_normalized",
         "start_location_normalized"]
    ]
)

In [ ]:
# ============================================================
# 12. ITINERARY: CLEAN ACTIVITIES
# ============================================================

def split_activity_text(value):
    if pd.isna(value):
        return []

    return [
        x.strip()
        for x in str(value).split(";")
        if x.strip()
    ]

itinerary["activity_list"] = itinerary["activities"].map(split_activity_text)
itinerary["activity_type_list"] = itinerary["activity_type"].map(split_activity_text)

itinerary["activity_count"] = itinerary["activity_list"].map(len)

# Count activity types.
def count_activity_type(types, target):
    return sum(
        1 for x in types
        if target in str(x).strip().lower()
    )

itinerary["major_sightseeing_count"] = itinerary["activity_type_list"].map(
    lambda x: count_activity_type(x, "major_sightseeing")
)

itinerary["photo_stop_count"] = itinerary["activity_type_list"].map(
    lambda x: count_activity_type(x, "photo_stop")
)

itinerary["shopping_stop_count"] = itinerary["activity_type_list"].map(
    lambda x: count_activity_type(x, "shopping_stop")
)

itinerary["free_time_count"] = itinerary["activity_type_list"].map(
    lambda x: count_activity_type(x, "free_time")
)

itinerary["transit_activity_count"] = itinerary["activity_type_list"].map(
    lambda x: count_activity_type(x, "transit")
)

itinerary["extra_cost_activity_count"] = itinerary["activity_type_list"].map(
    lambda x: count_activity_type(x, "extra_cost_addon")
)

In [ ]:
# ============================================================
# 13. ITINERARY: MEAL FEATURES
# ============================================================

def normalize_meal_text(value):
    if pd.isna(value):
        return ""

    return str(value).strip().lower()

itinerary["meals_clean"] = itinerary["meals_included_today"].map(
    normalize_meal_text
)

itinerary["breakfast_included"] = itinerary["meals_clean"].str.contains(
    "breakfast", na=False
)

itinerary["lunch_included"] = itinerary["meals_clean"].str.contains(
    "lunch", na=False
)

itinerary["dinner_included"] = itinerary["meals_clean"].str.contains(
    "dinner", na=False
)

itinerary["meal_ambiguity_flag"] = itinerary["meals_clean"].str.contains(
    r"\+|or|either|ambiguous",
    regex=True,
    na=False
)

In [ ]:
# ============================================================
# 14. ITINERARY: DAY-LEVEL INTENSITY FEATURES
# ============================================================

# IMPORTANT:
# These are transparent heuristic features, NOT a trained ML model.
# Do not claim that the resulting pace is scientifically validated yet.

itinerary["distance_km_clean"] = pd.to_numeric(
    itinerary["distance_km"], errors="coerce"
)

itinerary["drive_count"] = pd.to_numeric(
    itinerary["stops_requiring_separate_drives"], errors="coerce"
).fillna(0)

itinerary["transit_hours_clean"] = itinerary["transit_hours_avg"].fillna(0)

# Simple transparent intensity points.
itinerary["intensity_points"] = (
    itinerary["transit_hours_clean"] * 2.0
    + itinerary["activity_count"] * 4.0
    + itinerary["drive_count"] * 3.0
    + itinerary["major_sightseeing_count"] * 2.0
    + itinerary["shopping_stop_count"] * 1.0
    - itinerary["free_time_count"] * 4.0
)

# Don't allow negative intensity.
itinerary["intensity_points"] = itinerary["intensity_points"].clip(lower=0)

print(
    itinerary[
        ["package_id", "day_number", "activity_count",
         "drive_count", "transit_hours_clean", "intensity_points"]
    ]
)

In [ ]:
# ============================================================
# 15. ITINERARY: FLAG DAYS FOR REVIEW
# ============================================================

# These thresholds are deliberately conservative and should be
# tuned after you collect a larger dataset.

itinerary["long_drive_flag"] = (
    itinerary["transit_hours_clean"] >= 6
)

itinerary["high_distance_flag"] = (
    itinerary["distance_km_clean"] >= 200
)

itinerary["many_activities_flag"] = (
    itinerary["activity_count"] >= 4
)

itinerary["extra_cost_flag"] = (
    itinerary["extra_cost_activity_count"] > 0
)

itinerary["verification_flag"] = (
    itinerary["flagged_for_verification"].notna()
)

itinerary["day_concern_count"] = (
    itinerary["long_drive_flag"].astype(int)
    + itinerary["high_distance_flag"].astype(int)
    + itinerary["many_activities_flag"].astype(int)
    + itinerary["extra_cost_flag"].astype(int)
    + itinerary["verification_flag"].astype(int)
)

In [ ]:
# ============================================================
# 16. BUILD PACKAGE-LEVEL ITINERARY FEATURES
# ============================================================

itinerary_summary = (
    itinerary
    .groupby("package_id", as_index=False)
    .agg(
        itinerary_days=("day_number", "nunique"),
        total_distance_km=("distance_km_clean", "sum"),
        total_transit_hours=("transit_hours_clean", "sum"),
        total_drive_count=("drive_count", "sum"),
        total_activity_count=("activity_count", "sum"),
        total_major_sightseeing=("major_sightseeing_count", "sum"),
        total_photo_stops=("photo_stop_count", "sum"),
        total_shopping_stops=("shopping_stop_count", "sum"),
        total_free_time=("free_time_count", "sum"),
        total_extra_cost_activities=("extra_cost_activity_count", "sum"),
        total_day_concerns=("day_concern_count", "sum"),
        max_day_intensity=("intensity_points", "max"),
        avg_day_intensity=("intensity_points", "mean"),
    )
)

# Number of days with explicit verification concerns.
verification_summary = (
    itinerary.groupby("package_id")["verification_flag"]
    .sum()
    .reset_index(name="verification_day_count")
)

itinerary_summary = itinerary_summary.merge(
    verification_summary,
    on="package_id",
    how="left"
)

print(itinerary_summary)

In [ ]:
# ============================================================
# 17. INFER A TEMPORARY ITINERARY PACE
# ============================================================

# This is only a heuristic derived from itinerary intensity.
# If the agency provides an official itinerary_pace, that value
# should remain separate and take precedence.

def infer_pace(score):
    if pd.isna(score):
        return pd.NA

    if score < 20:
        return "relaxed"
    elif score < 40:
        return "moderate"
    else:
        return "packed"

itinerary_summary["inferred_itinerary_pace"] = (
    itinerary_summary["avg_day_intensity"].map(infer_pace)
)

itinerary_summary[
    ["package_id", "avg_day_intensity", "inferred_itinerary_pace"]
]

In [ ]:
# ============================================================
# 18. ACCOMMODATION CLEANING
# ============================================================

accommodation["destination_normalized"] = accommodation[
    "destination"
].map(normalize_place)

accommodation["accommodation_category_clean"] = accommodation[
    "accommodation_category"
].map(clean_text)

accommodation["hotel_name_clean"] = accommodation[
    "hotel_name"
].map(clean_text)

# Hotel guarantee is already normalized to:
# true / false / unspecified

accommodation_summary = (
    accommodation
    .groupby("package_id", as_index=False)
    .agg(
        accommodation_count=("destination", "count"),
        accommodation_destinations=(
            "destination_normalized",
            lambda x: " | ".join(x.dropna().astype(str))
        ),
        hotel_names=(
            "hotel_name_clean",
            lambda x: " | ".join(x.dropna().astype(str))
        ),
        accommodation_categories=(
            "accommodation_category_clean",
            lambda x: " | ".join(x.dropna().astype(str))
        ),
        guaranteed_hotel_count=(
            "hotel_guaranteed",
            lambda x: (x == "true").sum()
        ),
        unguaranteed_hotel_count=(
            "hotel_guaranteed",
            lambda x: (x == "false").sum()
        ),
    )
)

print(accommodation_summary)

In [ ]:
# ============================================================
# 19. INCLUSIONS CLEANING
# ============================================================

inclusions["inclusion_item_clean"] = inclusions[
    "inclusion_item"
].map(clean_text)

# Remove duplicate inclusion items per package.
inclusions = inclusions.drop_duplicates(
    subset=["package_id", "inclusion_item_clean"]
)

inclusion_summary = (
    inclusions
    .groupby("package_id", as_index=False)
    .agg(
        inclusion_count=("inclusion_item_clean", "count"),
        inclusion_text=(
            "inclusion_item_clean",
            lambda x: " | ".join(x.dropna().astype(str))
        )
    )
)

print(inclusion_summary)

In [ ]:
# ============================================================
# 20. EXCLUSIONS CLEANING
# ============================================================

exclusions["exclusion_item_clean"] = exclusions[
    "exclusion_item"
].map(clean_text)

exclusions = exclusions.drop_duplicates(
    subset=["package_id", "exclusion_item_clean"]
)

exclusion_summary = (
    exclusions
    .groupby("package_id", as_index=False)
    .agg(
        exclusion_count=("exclusion_item_clean", "count"),
        exclusion_text=(
            "exclusion_item_clean",
            lambda x: " | ".join(x.dropna().astype(str))
        )
    )
)

print(exclusion_summary)

In [ ]:
# ============================================================
# 21. EXTRACT KNOWN TAX / GST INFORMATION
# ============================================================

def extract_gst(text):
    if pd.isna(text):
        return np.nan

    match = re.search(r"(\d+(?:\.\d+)?)\s*%\s*gst", str(text).lower())

    if match:
        return float(match.group(1))

    return np.nan

exclusions["gst_percent_extracted"] = exclusions[
    "exclusion_item_clean"
].map(extract_gst)

gst_summary = (
    exclusions
    .groupby("package_id", as_index=False)["gst_percent_extracted"]
    .max()
    .rename(columns={"gst_percent_extracted": "gst_percent"})
)

print(gst_summary)

In [ ]:
# ============================================================
# 22. BUILD PACKAGE-LEVEL COST FEATURES
# ============================================================

packages["price"] = pd.to_numeric(packages["price"], errors="coerce")

# Base price per day / night.
packages["price_per_day"] = (
    packages["price"] / packages["duration_days"]
)

packages["price_per_night"] = (
    packages["price"] / packages["duration_nights"]
)

# Number of destinations.
packages["destination_count"] = packages[
    "destination_list_normalized"
].map(len)

# Number of themes.
packages["theme_count"] = packages["theme_list"].map(len)

In [ ]:
# ============================================================
# 23. MERGE ALL CHILD-TABLE FEATURES INTO PACKAGES
# ============================================================

processed_packages = packages.copy()

processed_packages = processed_packages.merge(
    itinerary_summary,
    on="package_id",
    how="left"
)

processed_packages = processed_packages.merge(
    accommodation_summary,
    on="package_id",
    how="left"
)

processed_packages = processed_packages.merge(
    inclusion_summary,
    on="package_id",
    how="left"
)

processed_packages = processed_packages.merge(
    exclusion_summary,
    on="package_id",
    how="left"
)

processed_packages = processed_packages.merge(
    gst_summary,
    on="package_id",
    how="left"
)

# If an official pace is missing, retain the inferred pace separately.
processed_packages["official_itinerary_pace"] = (
    processed_packages["itinerary_pace"]
)

processed_packages["effective_itinerary_pace"] = (
    processed_packages["itinerary_pace"]
    .fillna(processed_packages["inferred_itinerary_pace"])
)

print(processed_packages.shape)
print(processed_packages.head())

In [ ]:
# ============================================================
# 24. PACKAGE-LEVEL COST / VALUE FEATURES
# ============================================================

# Known GST percentage.
processed_packages["gst_percent"] = pd.to_numeric(
    processed_packages["gst_percent"], errors="coerce"
)

# Conservative "known tax amount".
# This does NOT estimate entrance fees, personal expenses, etc.
# because their actual amount is not provided.
processed_packages["known_gst_amount"] = (
    processed_packages["price"] *
    processed_packages["gst_percent"].fillna(0) / 100
)

processed_packages["price_plus_known_gst"] = (
    processed_packages["price"] +
    processed_packages["known_gst_amount"]
)

processed_packages["known_additional_cost_ratio"] = (
    processed_packages["known_gst_amount"] /
    processed_packages["price"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Count exclusions that may represent additional traveler costs.
COST_KEYWORDS = [
    "cost",
    "fee",
    "fees",
    "charges",
    "gst",
    "entrance",
    "parking",
    "permit",
    "personal expenses",
    "adventure activities",
    "insurance",
    "tips",
    "luggage",
    "extra meals",
]

def count_cost_related_exclusions(text):
    if pd.isna(text):
        return 0

    text = str(text).lower()

    return sum(
        1 for keyword in COST_KEYWORDS
        if keyword in text
    )

processed_packages["cost_related_exclusion_signal"] = (
    processed_packages["exclusion_text"]
    .map(count_cost_related_exclusions)
)

In [ ]:
# ============================================================
# 25. BUILD CANONICAL PACKAGE TEXT FOR NLP / EMBEDDINGS
# ============================================================

def safe_text(value):
    if pd.isna(value):
        return ""

    return str(value).strip()

def list_to_text(value):
    if isinstance(value, list):
        return ", ".join([str(x) for x in value if str(x).strip()])
    return safe_text(value)

def build_package_text(row):
    sections = []

    sections.append(f"Package: {safe_text(row.get('package_name'))}")
    sections.append(f"Agency: {safe_text(row.get('agency_name'))}")
    sections.append(f"Origin: {safe_text(row.get('start_location_normalized'))}")

    sections.append(
        f"Destinations: {list_to_text(row.get('destination_list_normalized'))}"
    )

    sections.append(
        f"Duration: {safe_text(row.get('duration_days'))} days, "
        f"{safe_text(row.get('duration_nights'))} nights"
    )

    sections.append(
        f"Price: INR {safe_text(row.get('price'))}"
    )

    sections.append(
        f"Transport: {safe_text(row.get('transport_type'))}"
    )

    sections.append(
        f"Themes: {list_to_text(row.get('theme_list'))}"
    )

    sections.append(
        f"Suitable for: {safe_text(row.get('suited_for'))}"
    )

    sections.append(
        f"Itinerary pace: {safe_text(row.get('effective_itinerary_pace'))}"
    )

    sections.append(
        f"Accommodation: {safe_text(row.get('accommodation_categories'))}; "
        f"Hotels: {safe_text(row.get('hotel_names'))}"
    )

    sections.append(
        f"Inclusions: {safe_text(row.get('inclusion_text'))}"
    )

    sections.append(
        f"Exclusions: {safe_text(row.get('exclusion_text'))}"
    )

    sections.append(
        f"Itinerary summary: "
        f"{safe_text(row.get('itinerary_days'))} itinerary days, "
        f"{safe_text(row.get('total_distance_km'))} km total stated distance, "
        f"{safe_text(row.get('total_transit_hours'))} total transit hours, "
        f"{safe_text(row.get('total_activity_count'))} activities."
    )

    return " ".join([s for s in sections if s.strip()])

processed_packages["canonical_package_text"] = (
    processed_packages.apply(build_package_text, axis=1)
)

print(processed_packages[
    ["package_id", "canonical_package_text"]
].to_string(index=False))

In [ ]:
# ============================================================
# 26. OPTIONAL: CREATE DAY-LEVEL CANONICAL ITINERARY TEXT
# ============================================================

def build_day_text(row):
    parts = [
        f"Day {safe_text(row.get('day_number'))}",
        f"Stops: {safe_text(row.get('stops'))}",
        f"Activities: {safe_text(row.get('activities'))}",
        f"Activity types: {safe_text(row.get('activity_type'))}",
        f"Distance: {safe_text(row.get('distance_km_clean'))} km",
        f"Transit: {safe_text(row.get('transit_hours'))}",
        f"Meals: {safe_text(row.get('meals_included_today'))}",
        f"Notes: {safe_text(row.get('notes'))}",
        f"Verification: {safe_text(row.get('flagged_for_verification'))}",
    ]

    return " | ".join([p for p in parts if p.split(": ", 1)[-1].strip()])

itinerary["canonical_day_text"] = itinerary.apply(
    build_day_text,
    axis=1
)

itinerary_package_text = (
    itinerary
    .sort_values(["package_id", "day_number"])
    .groupby("package_id")["canonical_day_text"]
    .apply(lambda x: " ".join(x))
    .reset_index(name="canonical_itinerary_text")
)

processed_packages = processed_packages.merge(
    itinerary_package_text,
    on="package_id",
    how="left"
)

processed_packages["canonical_package_text"] = (
    processed_packages["canonical_package_text"].fillna("")
    + " Itinerary details: "
    + processed_packages["canonical_itinerary_text"].fillna("")
)

In [ ]:
# ============================================================
# 27. FINAL COLUMN CLEANUP
# ============================================================

# Convert list columns into pipe-separated strings for CSV storage.
LIST_COLUMNS = [
    "destination_list",
    "theme_list",
    "transport_list",
    "destination_list_normalized",
]

for col in LIST_COLUMNS:
    if col in processed_packages.columns:
        processed_packages[col] = processed_packages[col].map(
            lambda x: " | ".join(map(str, x)) if isinstance(x, list) else x
        )

# Round derived continuous features for readability.
ROUND_COLUMNS = [
    "price_per_day",
    "price_per_night",
    "known_gst_amount",
    "price_plus_known_gst",
    "known_additional_cost_ratio",
    "total_distance_km",
    "total_transit_hours",
    "avg_day_intensity",
    "max_day_intensity",
]

for col in ROUND_COLUMNS:
    if col in processed_packages.columns:
        processed_packages[col] = pd.to_numeric(
            processed_packages[col], errors="coerce"
        ).round(2)

print(processed_packages.dtypes)

In [ ]:
# ============================================================
# 28. DATA VALIDATION
# ============================================================

print("========== DATA QUALITY REPORT ==========")

# 1. Package IDs
print("\nDuplicate package IDs:")
print(processed_packages[
    processed_packages["package_id"].duplicated(keep=False)
][["package_id", "package_name"]])

# 2. Missing critical fields
critical_columns = [
    "package_id",
    "package_name",
    "agency_name",
    "destinations",
    "start_location",
    "duration_days",
    "duration_nights",
    "price",
]

print("\nMissing critical values:")
for col in critical_columns:
    if col in processed_packages.columns:
        print(
            f"{col}: "
            f"{processed_packages[col].isna().sum()} missing"
        )

# 3. Invalid numeric values
print("\nInvalid / non-positive prices:")
print(
    processed_packages[
        processed_packages["price"].isna() |
        (processed_packages["price"] <= 0)
    ][["package_id", "price"]]
)

print("\nInvalid durations:")
print(
    processed_packages[
        processed_packages["duration_days"].isna() |
        (processed_packages["duration_days"] <= 0)
    ][["package_id", "duration_days"]]
)

# 4. Foreign keys in child tables
valid_ids = set(processed_packages["package_id"].dropna())

for name, df in {
    "itinerary": itinerary,
    "accommodation": accommodation,
    "inclusions": inclusions,
    "exclusions": exclusions,
}.items():

    if "package_id" in df.columns:
        orphan_ids = set(df["package_id"].dropna()) - valid_ids
        print(f"{name} orphan package IDs:", orphan_ids)

print("\n========== END REPORT ==========")

In [ ]:
# ============================================================
# 29. SAVE PROCESSED DATA
# ============================================================

# Main package-level dataset.
processed_packages.to_csv(
    OUTPUT_DIR / "packages_processed.csv",
    index=False
)

# Child-level datasets are also preserved.
itinerary.to_csv(
    OUTPUT_DIR / "itinerary_processed.csv",
    index=False
)

accommodation.to_csv(
    OUTPUT_DIR / "accommodation_processed.csv",
    index=False
)

inclusions.to_csv(
    OUTPUT_DIR / "inclusions_processed.csv",
    index=False
)

exclusions.to_csv(
    OUTPUT_DIR / "exclusions_processed.csv",
    index=False
)

# Embedding-ready file.
embedding_columns = [
    "package_id",
    "package_name",
    "agency_name",
    "price",
    "duration_days",
    "duration_nights",
    "destination_count",
    "theme_count",
    "effective_itinerary_pace",
    "canonical_package_text",
]

embedding_columns = [
    c for c in embedding_columns
    if c in processed_packages.columns
]

packages_for_embedding = processed_packages[embedding_columns].copy()

packages_for_embedding.to_csv(
    OUTPUT_DIR / "packages_for_embedding.csv",
    index=False
)

# Also save everything to one Excel workbook for inspection.
with pd.ExcelWriter(
    OUTPUT_DIR / "TripLens_processed_data.xlsx",
    engine="openpyxl"
) as writer:

    processed_packages.to_excel(
        writer, sheet_name="Packages_Processed", index=False
    )

    itinerary.to_excel(
        writer, sheet_name="Itinerary_Processed", index=False
    )

    accommodation.to_excel(
        writer, sheet_name="Accommodation_Processed", index=False
    )

    inclusions.to_excel(
        writer, sheet_name="Inclusions_Processed", index=False
    )

    exclusions.to_excel(
        writer, sheet_name="Exclusions_Processed", index=False
    )

print("Saved files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print(" -", file)

In [ ]:
# ============================================================
# 30. FINAL PREVIEW
# ============================================================

display(
    packages_for_embedding[
        [
            "package_id",
            "package_name",
            "price",
            "duration_days",
            "destination_count",
            "theme_count",
            "effective_itinerary_pace",
            "canonical_package_text",
        ]
    ]
)

## Output of this notebook

The most important output is:

`processed_data/packages_for_embedding.csv`

It contains a clean package-level representation ready for:

**Package text → embedding model → vector database → semantic candidate retrieval**

The structured columns in `packages_processed.csv` are used later for:

- budget fit
- duration fit
- destination matching
- theme/interest matching
- transport matching
- accommodation scoring
- itinerary intensity
- value scoring
- trust/review features

### Do not remove the raw workbook

Keep the original source data unchanged. The recommended pipeline is:

Raw Google Sheet / Excel
→ this preprocessing notebook
→ processed CSV/Excel
→ embeddings
→ vector database
→ hybrid ranking